# Merge AmbigQA + SQuAD 2.0 + HaluBench for annotation
Run the cells top to bottom. At the end you get `merged_for_annotation.xlsx` (with Yes/No dropdowns) and a CSV.

In [ ]:
!pip install -q -U datasets pandas openpyxl

## 1. Config

In [ ]:
SAMPLE_PER_SOURCE = 500      # rows per dataset. Use None for everything (SQuAD train is ~130k rows!)
SEED = 42
AMBIG_SPLIT = "validation"   # "train" or "validation"
SQUAD_SPLIT = "validation"   # "train" or "validation"
HALU_SPLIT  = "test"         # HaluBench only has a test split

OUT_CSV  = "merged_for_annotation.csv"
OUT_XLSX = "merged_for_annotation.xlsx"

## 2. Loaders

In [ ]:
import json
import pandas as pd
from datasets import load_dataset

def load_any(names, *args, **kwargs):
    """Try several repo ids (new name first, old name as fallback)."""
    last_err = None
    for name in names:
        try:
            return load_dataset(name, *args, **kwargs)
        except Exception as e:
            last_err = e
            print(f"[warn] could not load '{name}': {str(e)[:150]}")
    raise last_err

def sample(df):
    if SAMPLE_PER_SOURCE and len(df) > SAMPLE_PER_SOURCE:
        return df.sample(SAMPLE_PER_SOURCE, random_state=SEED)
    return df

def first_str(x):
    """Return the first string from a str / list / nested list."""
    while isinstance(x, (list, tuple)):
        if len(x) == 0:
            return ""
        x = x[0]
    return "" if x is None else str(x)

def normalize_ambig_ann(ann):
    """AmbigQA 'annotations' can come as dict-of-lists or list-of-dicts; return dict-of-lists."""
    if isinstance(ann, list):
        return {
            "type":    [a.get("type") for a in ann],
            "answer":  [a.get("answer") for a in ann],
            "qaPairs": [a.get("qaPairs") for a in ann],
        }
    return ann

In [ ]:
def build_ambigqa():
    ds = load_any(["sewon/ambig_qa", "ambig_qa"], "light", split=AMBIG_SPLIT)
    rows = []
    for ex in ds:
        ann = normalize_ambig_ann(ex["annotations"])
        a_type = first_str(ann["type"])            # singleAnswer / multipleQAs

        if a_type == "singleAnswer":
            ans_list = ann["answer"][0]
            if not isinstance(ans_list, list):
                ans_list = [ans_list]
            answer = " | ".join(str(a) for a in ans_list)
            disamb = ""
        else:
            qa = ann["qaPairs"][0]
            if isinstance(qa, dict):                 # dict of lists
                qs, ans = qa["question"], qa["answer"]
            else:                                    # list of dicts
                qs  = [p["question"] for p in qa]
                ans = [p["answer"] for p in qa]
            answer = " | ".join(first_str(a) for a in ans)
            disamb = json.dumps(
                [{"question": q, "answer": a} for q, a in zip(qs, ans)],
                ensure_ascii=False, default=list,
            )

        rows.append({
            "id": f"ambigqa_{ex['id']}",
            "source": "ambigqa",
            "context": "",                           # open-domain: no passage
            "question": ex["question"],
            "answer": answer,
            "ref_original_label": "",
            "ref_ambigqa_type": a_type,
            "ref_ambigqa_disambiguated": disamb,
            "ref_squad_unanswerable": "",
        })
    return sample(pd.DataFrame(rows))


def build_squad():
    ds = load_any(["rajpurkar/squad_v2", "squad_v2"], split=SQUAD_SPLIT)
    rows = []
    for ex in ds:
        texts = ex["answers"]["text"]
        rows.append({
            "id": f"squad_{ex['id']}",
            "source": "squad_v2",
            "context": ex["context"],
            "question": ex["question"],
            "answer": texts[0] if len(texts) else "",   # empty = unanswerable
            "ref_original_label": "",
            "ref_ambigqa_type": "",
            "ref_ambigqa_disambiguated": "",
            "ref_squad_unanswerable": "Yes" if len(texts) == 0 else "No",
        })
    return sample(pd.DataFrame(rows))


def build_halubench():
    ds = load_any(["PatronusAI/HaluBench"], split=HALU_SPLIT)
    rows = []
    for i, ex in enumerate(ds):
        rows.append({
            "id": f"halubench_{ex.get('id', i)}",
            "source": f"halubench_{ex.get('source_ds', 'unknown')}",
            "context": ex.get("context", ""),
            "question": ex.get("question", ""),
            "answer": ex.get("answer", ""),
            "ref_original_label": ex.get("label", ""),   # PASS / FAIL
            "ref_ambigqa_type": "",
            "ref_ambigqa_disambiguated": "",
            "ref_squad_unanswerable": "",
        })
    return sample(pd.DataFrame(rows))

## 3. Build, merge, add the annotation columns

In [ ]:
ambig = build_ambigqa();     print("AmbigQA:",   len(ambig))
squad = build_squad();       print("SQuAD 2.0:", len(squad))
halu  = build_halubench();   print("HaluBench:", len(halu))

merged = pd.concat([ambig, squad, halu], ignore_index=True)
merged = merged.sample(frac=1, random_state=SEED).reset_index(drop=True)   # shuffle

# --- columns to be labeled manually ---
merged["is_grammatically_correct"] = ""   # Yes / No
merged["is_ambiguous"]             = ""   # Yes / No
merged["ambiguity_sources"]        = ""   # fill only if is_ambiguous == Yes
merged["is_hallucinated"]          = ""   # Yes / No
merged["annotator"] = ""
merged["notes"]     = ""

merged = merged.fillna("").astype(str)
print(merged["source"].value_counts())
merged.head()

## 4. Save as CSV + Excel (with Yes/No dropdowns)

In [ ]:
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

merged.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

label_cols = ["is_grammatically_correct", "is_ambiguous", "is_hallucinated"]
to_fill = label_cols + ["ambiguity_sources"]
widths = {"context": 60, "question": 40, "answer": 40,
          "ref_ambigqa_disambiguated": 40, "ambiguity_sources": 35, "notes": 30}

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    merged.to_excel(writer, index=False, sheet_name="annotate")
    ws = writer.sheets["annotate"]
    cols = list(merged.columns)
    last = len(merged) + 1

    dv = DataValidation(type="list", formula1='"Yes,No"', allow_blank=True)
    ws.add_data_validation(dv)
    for c in label_cols:
        L = get_column_letter(cols.index(c) + 1)
        dv.add(f"{L}2:{L}{last}")

    yellow = PatternFill("solid", fgColor="FFF2CC")
    for idx, name in enumerate(cols, start=1):
        cell = ws.cell(row=1, column=idx)
        cell.font = Font(bold=True)
        if name in to_fill:
            cell.fill = yellow
        ws.column_dimensions[get_column_letter(idx)].width = widths.get(name, 18)

    for row in ws.iter_rows(min_row=2, max_row=last):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "D2"

print("Saved:", OUT_CSV, "and", OUT_XLSX)

## 5. Download (or save to Google Drive)

In [ ]:
from google.colab import files
files.download(OUT_XLSX)
files.download(OUT_CSV)

# --- Optional: save to Google Drive instead ---
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(OUT_XLSX, '/content/drive/MyDrive/')
# shutil.copy(OUT_CSV,  '/content/drive/MyDrive/')